<h1>🏗️ Biofilter — Platform: <code>platform_etl_status</code> and <code>platform_etl_packages</code></h1>

What ran to produce this bundle, and whether it holds up.

Platform reports describe the **bundle**, not the biology in it. They
take no input — there is nothing to ask about.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

# Results land here whatever directory the kernel was started in.
_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

### 2. The summary: one row per data source

In [ ]:
result = bf.report.run("platform_etl_status")
status = result.to_pandas()

print(f"{len(status)} data sources in bundle {result.provenance['bundle_id']}")
status.groupby(["branch", "pipeline_state"]).size()

### 3. `pipeline_state`, and why it is not a boolean

| state | meaning | `pipeline_ok` |
| --- | --- | --- |
| `ok` | every required stage ran, each on the previous one's output | true |
| `unverifiable` | every stage ran, no hashes to prove they belong together | true |
| `misaligned` | a stage ran on something else | false |
| `incomplete` | a required stage is missing | false |
| `never_run` | no packages at all | false |

`pipeline_ok` means **nothing is known to be wrong** — weaker than
"everything is proven right". `unverifiable` is the gap between them.

In [ ]:
status[~status["pipeline_ok"]][
    ["source_system", "data_source", "branch", "pipeline_state", "latest_error"]
]

### 4. Why this replaced a simple boolean

The report this replaces returned `pipeline_ok = False` for **51 of this
bundle's 68 sources**, and not one of them was broken.

The variant branch writes parquet straight from transform — there is no
load stage to miss. A report that flags a design decision as a failure
teaches people to ignore it.

In [ ]:
variant = status[status["branch"] == "variant"]

print(f"{len(variant)} variant sources")
print("with a load stage:", int(variant["load_package_id"].notna().sum()))
print("reported ok:", int(variant["pipeline_ok"].sum()))

### 5. `unverifiable` is not `misaligned`

Some DTPs read database state rather than a downloaded file, so they
produce no hash and alignment cannot be shown either way.
`transform_aligned` is **null** there rather than false — false reads as
"this is wrong" rather than "this is unproven".

In [ ]:
status[status["pipeline_state"] == "unverifiable"][
    ["data_source", "transform_aligned", "load_aligned", "pipeline_ok"]
]

### 6. A failure in the history, and a source that is fine anyway

`latest_error` is the most recent failure whether or not it was retried
successfully. `pipeline_state = 'ok'` together with a non-null
`latest_error` means "it worked, but not on the first try".

In [ ]:
status[status["latest_error"].notna()][
    ["data_source", "pipeline_state", "pipeline_ok", "latest_error"]
]

### 7. The packages behind the summary

`platform_etl_packages` is the unaggregated record: one row per package,
which is one stage of one run. Reach for it when the summary says
something surprising.

In [ ]:
packages = bf.report.run("platform_etl_packages").to_pandas()

print(f"{len(packages)} packages")
packages.groupby(["operation_type", "status"]).size()

### 8. What "aligned" actually means

Each stage is its own package, and the digest of the extract's output is
carried forward — it reappears as the transform's hash, then the load's.
Alignment means a stage ran on the previous one's output, not that two
unrelated digests happen to match.

In [ ]:
one = packages[packages["data_source"] == "biogrid"]

one[["package_id", "operation_type", "extract_hash", "transform_hash", "load_hash"]]

### 9. Failures stay in the record

`platform_etl_status` reports the latest **good** stage, so a source can
read `ok` while a failed package sits here. That is the pair to look at
together.

In [ ]:
failed = packages[packages["status"].str.contains("fail", case=False, na=False)]

failed[["package_id", "data_source", "operation_type", "status", "note"]]

### 10. Export

In [ ]:
for path in result.write(OUTPUT_DIR / "platform_etl_status.csv"):
    print(path)

### 11. The same thing on the command line

```bash
biofilter report run --report-name platform_etl_status --output status.csv

biofilter report run --report-name platform_etl_packages \\
    --param operation_type=load --output loads.csv
```